# Anonymize Previous Files

Take old research files and create subject level data.

## Libraries

In [1]:
from pathlib import Path
import pandas as pd
import hashlib
from difflib import SequenceMatcher
import re

# A secret salt makes the hashed worker ids impossible to check against a list of known worker ids.
# It is created once and kept in raw_data_nosync/ (not tracked by git). Keep a private copy: it is only
# needed to reproduce the same ids from the raw MTurk files, not to use the data.
import secrets
SALT_FILE = Path("raw_data_nosync/salt.txt")
if not SALT_FILE.exists():
    SALT_FILE.write_text(secrets.token_hex(32))
SALT = SALT_FILE.read_text().strip()

def anonymize(worker_id):
    return hashlib.sha256((SALT + str(worker_id)).encode()).hexdigest()[:10]

def extract_index(col_name):
    match = re.search(r'(\d+)$', col_name)
    return int(match.group(1)) if match else None

# -------------------
# core function
# -------------------

def process_mturk_file(filepath):
    # keep_default_na=False: "None" / "null" are real cue words, not missing values
    df = pd.read_csv(filepath, dtype=str, keep_default_na=False)
    
    # detect columns
    input_cols = [c for c in df.columns if c.startswith("Input.word")]
    answer_cols = [c for c in df.columns if c.startswith("Answer.word")]
    
    # sort them properly (word1, word2, ...)
    input_cols = sorted(input_cols, key=extract_index)
    answer_cols = sorted(answer_cols, key=extract_index)
    
    rows = []
    n_no_cue = 0
    
    for _, row in df.iterrows():
        worker_id = anonymize(row["WorkerId"])
        
        for input_col, answer_col in zip(input_cols, answer_cols):
            word = row.get(input_col)
            answer = row.get(answer_col)
            
            # Some HITs have fewer cues than answer boxes; an answer with no cue is dropped.
            word, answer = str(word).strip(), str(answer).strip()
            if not word:
                if answer:
                    n_no_cue += 1
                continue
            if not answer:
                continue
            
            rows.append({
                "subject_id": worker_id,
                "word": str(word).strip(),
                "answer": str(answer).strip(),
                "assignment_status": row.get("AssignmentStatus", "")   # Approved / Rejected
            })
    
    if n_no_cue:
        print(f"Dropped {n_no_cue} answers with no cue word (HIT had fewer cues than answer boxes): {Path(filepath).name}")
    
    return pd.DataFrame(rows)

def find_index(lines, keyword):
    for i, l in enumerate(lines):
        if keyword in l.lower():
            return i
    return None

def get_next_valid_lines(lines, start_idx, n=2):
    collected = []
    i = start_idx + 1
    
    while i < len(lines) and len(collected) < n:
        if lines[i].strip():
            collected.append(lines[i])
        i += 1
    
    return collected

def process_text_file(filepath, subject_id):
    rows = []
    
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        lines = [l.strip() for l in f.readlines()]
    
    # --- find sections ---
    practice_idx = find_index(lines, "practice")
    experimental_idx = find_index(lines, "experimental")
    
    # --- get practice lines (2 only) ---
    if practice_idx is not None:
        practice_lines = get_next_valid_lines(lines, practice_idx, 2)
    else:
        practice_lines = []
        print(f"⚠️ No Practice section: {filepath}")
    
    # --- get experimental lines ---
    if experimental_idx is not None:
        experimental_lines = [
            l for l in lines[experimental_idx + 1:] if l.strip()
        ]
    else:
        experimental_lines = []
        print(f"⚠️ No Experimental section: {filepath}")
    
    all_lines = practice_lines + experimental_lines
    
    # --- parse lines ---
    for line in all_lines:
        parts = line.split(" ", 1)
        
        if len(parts) < 2:
            continue
        
        word, answer = parts

        # "book bag" is the cue "bookbag" in the 2019 word list
        if word.lower() == "book" and answer.lower().startswith("bag "):
            word, answer = "bookbag", answer[4:]

        # some files (e.g., Montana) number each trial: "1 team Contains players..."
        if word.isdigit():
            parts = answer.split(" ", 1)
            if len(parts) < 2:
                continue
            word, answer = parts
        
        rows.append({
            "subject_id": subject_id,
            "word": word.strip(),
            "answer": answer.strip()
        })
    
    return rows

def process_all_subjects(root_dir, output_file):
    all_rows = []
    subject_counter = 1
    
    root = Path(root_dir)
    
    # handle both: parent dir OR single indv_subs folder
    if root.name.startswith("indv_subs_"):
        folders = [root]
    else:
        folders = sorted(root.glob("indv_subs_*"))
    
    for folder in folders:
        print(f"Processing folder: {folder}")
        
        for file in folder.glob("*.txt"):
            # print(f"  → {file.name}")
            
            rows = process_text_file(file, subject_counter)
            all_rows.extend(rows)
            
            subject_counter += 1
    
    df = pd.DataFrame(all_rows)
    df.to_csv(output_file, index=False)
    
    print(f"\n✅ Saved {len(df)} rows from {subject_counter-1} subjects")
    return df

def process_word_list_csv_file(filepath):
    df = pd.read_csv(filepath, header=None, names=["word", "answer"], dtype=str, keep_default_na=False)
    
    df["word"] = df["word"].astype(str).str.strip()
    df["answer"] = df["answer"].astype(str).str.strip()
    
    df = df[df["answer"] != ""]
    df = df[~df["answer"].isna()]
    
    df["subject_id"] = df.groupby("word").cumcount() + 1
    
    return df[["subject_id", "word", "answer"]]

def process_word_list_txt_file(filepath):
    rows = []
    
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        lines = [l.strip() for l in f.readlines()]
    
    for line in lines:
        if not line:
            continue
        
        parts = line.split(None, 1)  # split on first whitespace
        
        if len(parts) < 2:
            continue
        
        word, answer = parts
        
        rows.append({
            "word": word.strip(),
            "answer": answer.strip()
        })
    
    df = pd.DataFrame(rows)
    
    # assign subject IDs per word
    df["subject_id"] = df.groupby("word").cumcount() + 1
    
    return df[["subject_id", "word", "answer"]]

def process_word_list_folder(folder_path, output_file=None, skip=()):
    all_rows = []
    subject_offset = 0  # ensures unique IDs across files
    
    folder = Path(folder_path)
    
    for file in sorted(folder.glob("*")):
        if file.name in skip:
            print(f"Skipping (processed separately): {file.name}")
            continue
        print(f"Processing: {file.name}")
        
        if file.suffix.lower() == ".csv":
            df = process_word_list_csv_file(file)
        
        elif file.suffix.lower() == ".txt":
            df = process_word_list_txt_file(file)
        
        else:
            print(f"  ⚠️ Skipping unsupported file: {file.name}")
            continue
        
        # shift subject IDs so they don't overlap across files
        df["subject_id"] += subject_offset
        
        subject_offset = df["subject_id"].max()
        
        all_rows.append(df)
    
    final_df = pd.concat(all_rows, ignore_index=True)
    
    if output_file:
        final_df.to_csv(output_file, index=False)
    
    print(f"\n✅ Total rows: {len(final_df)}")
    print(f"✅ Total subjects: {final_df['subject_id'].nunique()}")
    
    return final_df

def process_wide_wordlist(
    filepath,
    output_path=None,
    id_col="id",
    drop_cols=None
):
    df = pd.read_csv(filepath, dtype=str, keep_default_na=False)  # "none"/"null" are real cues/answers
    
    # --- columns to ignore ---
    drop_cols = drop_cols or []
    
    # ensure id exists
    if id_col not in df.columns:
        raise ValueError(f"Column '{id_col}' not found")
    
    # columns that are words
    word_cols = [
        c for c in df.columns
        if c != id_col and c not in drop_cols
    ]
    
    # --- reshape ---
    long_df = df.melt(
        id_vars=[id_col],
        value_vars=word_cols,
        var_name="word",
        value_name="answer"
    )
    
    # clean
    long_df["answer"] = long_df["answer"].astype(str).str.strip()
    
    # drop empty answers
    long_df = long_df[long_df["answer"] != ""]
    long_df = long_df[~long_df["answer"].isna()]
    
    # rename id → subject_id
    long_df = long_df.rename(columns={id_col: "subject_id"})
    
    # reorder
    long_df = long_df[["subject_id", "word", "answer"]]
    
    if output_path:
        long_df.to_csv(output_path, index=False)
    
    print(f"✅ Processed {len(long_df)} rows")
    return long_df

def process_paired_wide_no_id(filepath, output_path=None, start_id=1):
    df = pd.read_csv(filepath)
    
    # assign subject IDs
    df["subject_id"] = range(start_id, start_id + len(df))
    
    # --- find word/answer columns correctly ---
    word_cols = []
    answer_cols = []
    
    for col in df.columns:
        col_lower = col.lower()
        
        if "input.word" in col_lower:
            word_cols.append(col)
        elif "answer.word" in col_lower:
            answer_cols.append(col)
    
    # sort by number
    def extract_num(col):
        m = re.search(r'(\d+)', col)
        return int(m.group(1)) if m else 0
    
    word_cols = sorted(word_cols, key=extract_num)
    answer_cols = sorted(answer_cols, key=extract_num)
    
    # --- build long format ---
    rows = []
    
    for _, row in df.iterrows():
        subject_id = row["subject_id"]
        
        for w_col, a_col in zip(word_cols, answer_cols):
            word = row[w_col]
            answer = row[a_col]
            
            if pd.isna(word) or pd.isna(answer):
                continue
            
            word = str(word).strip()
            answer = str(answer).strip()
            
            if not word or not answer:
                continue
            
            rows.append({
                "subject_id": subject_id,
                "word": word,
                "answer": answer
            })
    
    long_df = pd.DataFrame(rows)
    
    if output_path:
        long_df.to_csv(output_path, index=False)
    
    print(f"✅ Processed {len(long_df)} rows from {len(df)} subjects")
    return long_df

# misspelled / variant cues -> the spelling used in the 2019 word list
CUE_FIXES = {"celcius": "celsius", "condfidence": "confidence", "don't": "dont", "innoncent": "innocent"}

# cues from the 2013 subject files that are not in the 2019 cue list; excluded
# ("cider": 16 participants wrote about cinder blocks; no raw file exists for it)
EXCLUDE_CUES = {"cider"}

def merge_all_processed_data(
    input_dir="raw_data",
    output_file="raw_data_all/all_data.csv"
):
    """Merge per-source files into one long file.

    Columns: subject_id (globally unique), file_subject_id (id within the source
    file), source_file, word, answer. Cue words are lowercased/stripped; empty
    answers and exact duplicate rows are dropped. Notes about IDs live in README.
    """
    all_dfs = []
    id_map = {}
    
    for file in sorted(Path(input_dir).glob("*.csv")):
        print(f"Processing: {file.name}")
        
        df = pd.read_csv(file, dtype=str, keep_default_na=False)
        
        if not {"subject_id", "word", "answer"}.issubset(df.columns):
            print(f"  ⚠️ Skipping (missing required columns): {file.name}")
            continue
        
        # --- clean ---
        df["word"] = df["word"].str.strip().str.lower().replace(CUE_FIXES)
        df["answer"] = df["answer"].str.strip()
        n0 = len(df)
        df = df[(df["answer"] != "") & (df["answer"].str.lower() != "nan") & (df["word"] != "")]
        df = df[~df["word"].isin(EXCLUDE_CUES)]
        n1 = len(df)
        df = df.drop_duplicates(["subject_id", "word", "answer"])
        print(f"  dropped {n0 - n1} empty, {n1 - len(df)} duplicate rows")
        
        # --- ids: keep original within-file id, add globally unique id ---
        # MTurk files share hashed WorkerIds, so the same worker keeps one id
        # across mturk_file_*.csv; every other source is unique within its file.
        if "assignment_status" not in df.columns:
            df["assignment_status"] = ""
        df["file_subject_id"] = df["subject_id"]
        group = "mturk" if file.name.startswith("mturk_file_") else file.name
        new_ids = []
        for old_id in df["file_subject_id"].unique():
            key = (group, old_id)
            if key not in id_map:
                id_map[key] = len(id_map) + 1
            new_ids.append((old_id, id_map[key]))
        df["subject_id"] = df["file_subject_id"].map(dict(new_ids))
        
        df["source_file"] = file.name
        all_dfs.append(df[["subject_id", "file_subject_id", "source_file", "word", "answer", "assignment_status"]])
    
    final_df = pd.concat(all_dfs, ignore_index=True)
    
    Path(output_file).parent.mkdir(exist_ok=True)
    final_df.to_csv(output_file, index=False)
    
    print("\n✅ FINAL SUMMARY")
    print(f"Rows: {len(final_df)}")
    print(f"Subjects: {final_df['subject_id'].nunique()}")
    print(f"Files merged: {len(all_dfs)}")
    
    return final_df

def _norm(text):
    return re.sub(r"\W+", " ", str(text).lower()).strip()

def _read_lines(path):
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return [l.strip() for l in f if l.strip()]

FUZZY_CUTOFF = 0.8  # tested against exact-aligned cues: ~96% precision, ~92% recall

def _best_ratio(text, candidates, cutoff):
    """Highest similarity between text and any candidate (0 if none reach cutoff)."""
    m = SequenceMatcher(autojunk=False)
    m.set_seq2(text)
    top = 0
    for c in candidates:
        if min(len(text), len(c)) / max(len(text), len(c), 1) < cutoff:
            continue
        m.set_seq1(c)
        if m.real_quick_ratio() < cutoff or m.quick_ratio() < cutoff:
            continue
        top = max(top, m.ratio())
        if top >= 0.999:
            break
    return top

ALIGN_CUTOFF = 0.5  # tested on cues with known answers: ~88% precision, ~92% recall

def _aligned_flags(raw_lines, kept_text, cutoff):
    """For cues whose _fixed file lost its line breaks: align the raw words to the kept text.
    A raw line is 'removed' if fewer than `cutoff` of its words appear in the aligned kept text."""
    raw_words, owner = [], []
    for i, line in enumerate(raw_lines):
        words = _norm(line).split()
        raw_words += words
        owner += [i] * len(words)
    kept_words = _norm(kept_text).split()
    matched = [0] * len(raw_lines)
    total = [0] * len(raw_lines)
    for o in owner:
        total[o] += 1
    for a, b, size in SequenceMatcher(None, raw_words, kept_words, autojunk=False).get_matching_blocks():
        for k in range(a, a + size):
            matched[owner[k]] += 1
    return [total[i] > 0 and matched[i] / total[i] < cutoff for i in range(len(raw_lines))]

def build_wiki_lookup(wn2_dir):
    """Per cue, find which raw answers the original (Word-Norms-2) cleaning removed.

    Compares line i of `0.5 raw words` with line i of `1 spell checked`; the line
    was removed if its spell-checked version is missing from `1.5 raw words no wiki/_fixed`.
    Only possible when raw and spell-checked have the same number of lines.
    Returns {cue: (status, {normalized raw answer: [removed, ...]})}.
    """
    wn2 = Path(wn2_dir)
    raw_d, sp_d, fx_d = wn2/"0.5 raw words", wn2/"1 spell checked", wn2/"1.5 raw words no wiki"/"_fixed"
    lookup = {}
    for raw_f in sorted(raw_d.glob("*.txt")):
        cue = raw_f.stem.replace("word_", "", 1) if raw_f.stem.startswith("word_") else raw_f.stem
        # some cues were saved as "<cue> R.txt" in the spell-checked and _fixed folders
        sp_f, fx_f = sp_d/raw_f.name, fx_d/raw_f.name
        # (and "dont" is "don't.txt" in those folders)
        alt = {"dont": "don't"}.get(raw_f.stem, raw_f.stem)
        if not sp_f.exists():
            sp_f = next((q for q in (sp_d/f"{raw_f.stem} R.txt", sp_d/f"{alt}.txt") if q.exists()), sp_f)
        if not fx_f.exists():
            fx_f = next((q for q in (fx_d/f"{raw_f.stem} R.txt", fx_d/f"{alt}.txt") if q.exists()), fx_f)
        if not sp_f.exists() or not fx_f.exists():
            lookup[cue] = ("no_files", {}); continue
        raw, sp, fx = _read_lines(raw_f), _read_lines(sp_f), _read_lines(fx_f)
        if len(raw) >= 20 and len(fx) <= 3:
            # _fixed lost its line breaks (a few giant lines); recover them by aligning with the raw lines.
            # (Cues where the original cleaning simply removed most lines, e.g. 9 of 54, use the line comparison below.)
            flags = {}
            for r, removed in zip(raw, _aligned_flags(raw, " ".join(fx), ALIGN_CUTOFF)):
                flags.setdefault(_norm(r), []).append(removed)
            lookup[cue] = ("aligned", flags); continue
        if len(raw) != len(sp):
            # line counts differ: a raw line counts as removed if nothing in _fixed is similar to it
            fx_norm = [_norm(x) for x in fx]
            flags = {}
            for r in raw:
                n = _norm(r)
                flags.setdefault(n, []).append(_best_ratio(n, fx_norm, FUZZY_CUTOFF) < FUZZY_CUTOFF)
            lookup[cue] = ("fuzzy", flags); continue
        kept, flags = set(fx), {}
        for r, s in zip(raw, sp):
            flags.setdefault(_norm(r), []).append(s not in kept)
        lookup[cue] = ("exact", flags)
    return lookup

def add_wiki_flag(df, wn2_dir):
    """wiki_flag: True = removed by original cleaning (exclude later); False = kept;
    blank = unknown (see wiki_method)."""
    lookup = build_wiki_lookup(wn2_dir)
    flags, methods = [], []
    for cue, ans in zip(df["word"], df["answer"]):
        status, table = lookup.get(cue, ("no_files", {}))
        if status not in ("exact", "fuzzy", "aligned"):
            flags.append(pd.NA); methods.append(status if cue in lookup else "cue_not_in_raw"); continue
        key = _norm(ans)
        hits = table.get(key)
        how = status
        if hits is None:
            # the 2019 "raw" files were lightly cleaned (commas removed, some typos fixed),
            # so fall back to the most similar raw line for that cue
            best_key, best = None, 0
            m = SequenceMatcher(autojunk=False); m.set_seq2(key)
            for k in table:
                if min(len(k), len(key)) / max(len(k), len(key), 1) < FUZZY_CUTOFF:
                    continue
                m.set_seq1(k)
                if m.real_quick_ratio() < FUZZY_CUTOFF or m.quick_ratio() < FUZZY_CUTOFF:
                    continue
                r = m.ratio()
                if r > best:
                    best_key, best = k, r
            if best >= FUZZY_CUTOFF:
                hits, how = table[best_key], status + "_text_similar"
        if hits is None:
            flags.append(pd.NA); methods.append("answer_not_in_raw")
        else:
            flags.append(all(hits)); methods.append(how)
    out = df.copy()
    out["wiki_flag"] = pd.array(flags, dtype="boolean")
    out["wiki_method"] = methods
    # no original decision available: treat as kept (the text rules in wiki_rule still apply)
    out["wiki_flag"] = out["wiki_flag"].fillna(False)
    return out

_RULE_NUMBERED = re.compile(r"(?:^|\s)1[\.\):]?\s+\S.{8,}?\s2[\.\):]?\s+\S")
_RULE_POS = re.compile(r"\((?:n|v|adj|adv|noun|verb|adjective|adverb)\.?\)|^\s*(?:noun|verb|adjective|adverb)\s*[:\-]|\bnoun:|\bverb:", re.I)
_RULE_WEB = re.compile(r"wikipedia|https?://|www\.|\[\d+\]|\bretrieved\b", re.I)
_RULE_ARTICLE = re.compile(r"may refer to|disambiguation|this article (is|has|needs)|for other uses|(^|\s)a\.\s.+\sb\.\s", re.I)

def wiki_rule_reasons(answer):
    """Text patterns typical of pasted dictionary / web-page definitions."""
    a = str(answer)
    reasons = []
    if len(a) > 500:
        reasons.append("very_long")                # reviewed by hand: all >500 chars are pasted text
    if len(a) > 60 and _RULE_NUMBERED.search(a):
        reasons.append("numbered_senses")          # "1 the state of ... 2 a person ..."
    parts = [p for p in a.split(";") if p.strip()]
    if len(parts) >= 3 and sum(len(p) for p in parts) / len(parts) >= 40:
        reasons.append("semicolon_definition")     # long clauses separated by ";"
    if len(a) > 80 and _RULE_POS.search(a):
        reasons.append("pos_tag")                  # "(noun)", "verb: ..."
    if _RULE_WEB.search(a):
        reasons.append("web_text")
    if _RULE_ARTICLE.search(a):
        reasons.append("article_text")                # "may refer to", "(disambiguation)", "a. ... b. ..." senses
    return reasons

def add_wiki_rule(df):
    """wiki_rule: comma-separated reasons an answer looks like pasted definition text
    (blank = no pattern). Independent of wiki_flag; it does not change it."""
    out = df.copy()
    out["wiki_rule"] = [",".join(wiki_rule_reasons(a)) for a in out["answer"]]
    # every answer from an MTurk assignment that was rejected (wiki copies, "I don't know", paragraphs)
    rejected = out["assignment_status"] == "Rejected"
    out.loc[rejected, "wiki_rule"] = out.loc[rejected, "wiki_rule"].map(lambda r: ",".join(filter(None, [r, "rejected_hit"])))
    # the same long answer (>40 characters) twice within a cue is not chance: keep the
    # first copy, mark later copies. Short repeated answers ("money") are kept.
    key = out["word"] + "||" + out["answer"].map(_norm)
    later_copy = key.duplicated(keep="first") & (out["answer"].map(_norm).str.len() > 40)
    out.loc[later_copy, "wiki_rule"] = out.loc[later_copy, "wiki_rule"].map(lambda r: ",".join(filter(None, [r, "long_duplicate"])))
    # cues that are not in the 2019 cue list are stray (e.g., wrapped answer lines read as cues)
    stray = out["wiki_method"] == "cue_not_in_raw"
    out.loc[stray, "wiki_rule"] = out.loc[stray, "wiki_rule"].map(lambda r: ",".join(filter(None, [r, "stray_cue"])))
    return out


## Data

### Mturk Files 

Raw Mturk files with ids to clean up.

In [2]:
input_file = "raw_data_nosync/mturk_raw/mturk raw 1.csv"
output_file = "raw_data/mturk_file_1.csv"
    
Path("processed").mkdir(exist_ok=True)
    
df = process_mturk_file(input_file)
df.to_csv(output_file, index=False)
    
print(f"Saved {len(df)} rows to {output_file}")

Saved 5925 rows to raw_data/mturk_file_1.csv


In [3]:
input_file = "raw_data_nosync/mturk_raw/mturk raw 2.csv"
output_file = "raw_data/mturk_file_2.csv"
    
Path("processed").mkdir(exist_ok=True)
    
df = process_mturk_file(input_file)
df.to_csv(output_file, index=False)
    
print(f"Saved {len(df)} rows to {output_file}")

Saved 1579 rows to raw_data/mturk_file_2.csv


In [4]:
input_file = "raw_data_nosync/mturk_raw/mturk raw 3.csv"
output_file = "raw_data/mturk_file_3.csv"
    
Path("processed").mkdir(exist_ok=True)
    
df = process_mturk_file(input_file)
df.to_csv(output_file, index=False)
    
print(f"Saved {len(df)} rows to {output_file}")

Saved 2644 rows to raw_data/mturk_file_3.csv


In [5]:
input_file = "raw_data_nosync/mturk_raw/mturk raw 4.csv"
output_file = "raw_data/mturk_file_4.csv"
    
Path("processed").mkdir(exist_ok=True)
    
df = process_mturk_file(input_file)
df.to_csv(output_file, index=False)
    
print(f"Saved {len(df)} rows to {output_file}")

Saved 360 rows to raw_data/mturk_file_4.csv


In [6]:
input_file = "raw_data_nosync/mturk_raw/mturk raw 5.csv"
output_file = "raw_data/mturk_file_5.csv"
    
Path("processed").mkdir(exist_ok=True)
    
df = process_mturk_file(input_file)
df.to_csv(output_file, index=False)
    
print(f"Saved {len(df)} rows to {output_file}")

Saved 300 rows to raw_data/mturk_file_5.csv


In [7]:
input_file = "raw_data_nosync/mturk_raw/mturk raw 6.csv"
output_file = "raw_data/mturk_file_6.csv"
    
Path("processed").mkdir(exist_ok=True)
    
df = process_mturk_file(input_file)
df.to_csv(output_file, index=False)
    
print(f"Saved {len(df)} rows to {output_file}")

Saved 395 rows to raw_data/mturk_file_6.csv


In [8]:
input_file = "raw_data_nosync/mturk_raw/mturk raw 7.csv"
output_file = "raw_data/mturk_file_7.csv"
    
Path("processed").mkdir(exist_ok=True)
    
df = process_mturk_file(input_file)
df.to_csv(output_file, index=False)
    
print(f"Saved {len(df)} rows to {output_file}")

Saved 2108 rows to raw_data/mturk_file_7.csv


In [9]:
input_file = "raw_data_nosync/mturk_raw/mturk raw 8.csv"
output_file = "raw_data/mturk_file_8.csv"
    
Path("processed").mkdir(exist_ok=True)
    
df = process_mturk_file(input_file)
df.to_csv(output_file, index=False)
    
print(f"Saved {len(df)} rows to {output_file}")

Saved 6974 rows to raw_data/mturk_file_8.csv


In [10]:
input_file = "raw_data_nosync/mturk_raw/mturk raw 9.csv"
output_file = "raw_data/mturk_file_9.csv"
    
Path("processed").mkdir(exist_ok=True)
    
df = process_mturk_file(input_file)
df.to_csv(output_file, index=False)
    
print(f"Saved {len(df)} rows to {output_file}")

Saved 4493 rows to raw_data/mturk_file_9.csv


### Individual Subject Files

In [11]:
process_all_subjects(
    root_dir="raw_data_nosync",
    output_file="raw_data/individual_subjects.csv"
)

Processing folder: raw_data_nosync/indv_subs_1


⚠️ No Practice section: raw_data_nosync/indv_subs_1/s43.txt
Processing folder: raw_data_nosync/indv_subs_10
Processing folder: raw_data_nosync/indv_subs_11
Processing folder: raw_data_nosync/indv_subs_2
Processing folder: raw_data_nosync/indv_subs_3


Processing folder: raw_data_nosync/indv_subs_4
Processing folder: raw_data_nosync/indv_subs_5
Processing folder: raw_data_nosync/indv_subs_6


Processing folder: raw_data_nosync/indv_subs_7


Processing folder: raw_data_nosync/indv_subs_8


Processing folder: raw_data_nosync/indv_subs_9



✅ Saved 46770 rows from 766 subjects


,subject_id,word,answer
0,1,finger,"foods, paint, hands"
1,1,careless,free flowing
2,1,casual,"laid back, calm, jeans and a tshirt"
3,1,calories,food
4,1,checking,check book
...,...,...,...
46765,766,cabbage,"greens, bowling, water, mom, cooking, cornbrea..."
46766,766,feet,"toes, socks, ankle, shoes, toenails, sandals"
46767,766,quiet,"shhhh!, no noise, quiet game, silence,"
46768,766,search,"find, look for, found, lost, wonder,"


### Process Long Files

### Word list 23b

The original word list 23 folder has one file with the answers for both halves (23a and 23b). `wl23b.csv` lists the 23b cues; their answers are pulled out of `transpose.txt` into `long_raw/word_list_23b.txt`.

In [12]:
orig = Path("raw_data_nosync/word_list_23_original")
cues_23b = {line.strip().lower() for line in open(orig / "wl23b.csv", errors="ignore") if line.strip()}

lines = []
for line in open(orig / "transpose.txt", errors="ignore"):
    parts = line.strip().split("\t", 1)
    if parts[0].lower() in cues_23b and len(parts) == 2 and parts[1].strip():
        lines.append(f"{parts[0]}\t{parts[1].strip()}")

Path("raw_data_nosync/long_raw/word_list_23b.txt").write_text("\n".join(lines) + "\n")
print(f"{len(lines):,} answers for {len(cues_23b)} cues written to word_list_23b.txt")

2,128 answers for 30 cues written to word_list_23b.txt


In [13]:
df = process_word_list_folder(
    folder_path="raw_data_nosync/long_raw",
    output_file="raw_data/word_long_all.csv",
    skip={"word_list_11.csv"}   # wide format, processed below
)

Processing: .DS_Store
  ⚠️ Skipping unsupported file: .DS_Store
Skipping (processed separately): word_list_11.csv
Processing: word_list_14.txt
Processing: word_list_15.txt
Processing: word_list_16.txt


Processing: word_list_17.txt
Processing: word_list_18.txt
Processing: word_list_19_montana.csv
Processing: word_list_1_sona.csv
Processing: word_list_20a.csv
Processing: word_list_20b.csv
Processing: word_list_21a.csv
Processing: word_list_21b.csv
Processing: word_list_22a.csv
Processing: word_list_22b.csv
Processing: word_list_23a.csv
Processing: word_list_23b.txt
Processing: word_list_24a.csv
Processing: word_list_24b.csv
Processing: word_list_25a.csv
Processing: word_list_25b.csv


Processing: word_list_26a.csv


Processing: word_list_2_sona.csv
Processing: word_list_3_sona.csv



✅ Total rows: 50092
✅ Total subjects: 1310


### Process Wide Files

In [14]:
# word_list_11 is a wide file (one column per cue; old Mac line endings) that sits in long_raw
df = process_wide_wordlist(
    "raw_data_nosync/long_raw/word_list_11.csv",
    output_path="raw_data/word_list_11_long.csv",
    id_col="id_code",
    drop_cols=["start_time","end_time","duration"]
)

✅ Processed 3720 rows


In [15]:
df = process_wide_wordlist(
    "raw_data_nosync/wide_raw/word_list_10.csv",
    output_path="raw_data/word_list_10_long.csv",
    id_col="id_code",                # or whatever it's called
    drop_cols=["start_time","end_time","duration"]      # optional
)

✅ Processed 3534 rows


In [16]:
df = process_wide_wordlist(
    "raw_data_nosync/wide_raw/word_list_12.csv",
    output_path="raw_data/word_list_12_long.csv",
    id_col="id_code",                # or whatever it's called
    drop_cols=["start_time","end_time","duration"]      # optional
)

✅ Processed 4712 rows


In [17]:
df = process_wide_wordlist(
    "raw_data_nosync/wide_raw/word_list_13.csv",
    output_path="raw_data/word_list_13_long.csv",
    id_col="id_code",                # or whatever it's called
    drop_cols=["start_time","end_time","duration"]      # optional
)

✅ Processed 4340 rows


In [18]:
df = process_paired_wide_no_id(
    "raw_data_nosync/mturk_extra/wl5 mturk.csv",
    output_path="raw_data/word_list_5_long.csv"
)

✅ Processed 52591 rows from 10523 subjects


## Put Together Data


In [19]:
df = merge_all_processed_data(
    input_dir="raw_data",
    output_file="raw_data_all/all_data.csv"
)


Processing: individual_subjects.csv
  dropped 16 empty, 121 duplicate rows
Processing: mturk_file_1.csv
  dropped 0 empty, 0 duplicate rows
Processing: mturk_file_2.csv
  dropped 0 empty, 0 duplicate rows
Processing: mturk_file_3.csv
  dropped 0 empty, 0 duplicate rows
Processing: mturk_file_4.csv
  dropped 0 empty, 0 duplicate rows
Processing: mturk_file_5.csv
  dropped 0 empty, 0 duplicate rows
Processing: mturk_file_6.csv
  dropped 0 empty, 0 duplicate rows
Processing: mturk_file_7.csv
  dropped 0 empty, 0 duplicate rows
Processing: mturk_file_8.csv
  dropped 0 empty, 0 duplicate rows
Processing: mturk_file_9.csv
  dropped 0 empty, 0 duplicate rows
Processing: word_list_10_long.csv
  dropped 0 empty, 0 duplicate rows
Processing: word_list_11_long.csv
  dropped 0 empty, 0 duplicate rows
Processing: word_list_12_long.csv
  dropped 0 empty, 0 duplicate rows
Processing: word_list_13_long.csv
  dropped 0 empty, 0 duplicate rows
Processing: word_list_5_long.csv
  dropped 0 empty, 0 duplic

  dropped 0 empty, 0 duplicate rows



✅ FINAL SUMMARY
Rows: 190400
Subjects: 13643
Files merged: 16


## Flag Wikipedia / dictionary answers

Flags answers removed by the original Word-Norms-2 cleaning (so they can be excluded from later processing). Rows stay in the data.

In [20]:
WN2_DIR = "/Users/erinbuchanan/GitHub/Research/3_published/46 Word-Norms-2"

all_data = pd.read_csv("raw_data_all/all_data.csv", dtype=str, keep_default_na=False)
all_data = add_wiki_flag(all_data, WN2_DIR)
all_data = add_wiki_rule(all_data)
all_data.to_csv("raw_data_all/all_data.csv", index=False)

print(all_data.groupby(["source_file", "wiki_method"]).size().unstack(fill_value=0))
print(all_data["wiki_flag"].value_counts(dropna=False))
print(pd.crosstab(all_data["wiki_flag"].astype(str), all_data["wiki_rule"] != ""))

wiki_method              aligned  answer_not_in_raw  cue_not_in_raw  exact  \
source_file                                                                  
individual_subjects.csv        0               6500              29  13396   
mturk_file_1.csv               0                  5               0   5785   
mturk_file_2.csv               0                  0               0    103   
mturk_file_3.csv               0                 45               0     44   
mturk_file_4.csv               0                  0               0    329   
mturk_file_5.csv               0                  0               0    287   
mturk_file_6.csv               0                  0               0    365   
mturk_file_7.csv               0                  1               0   1967   
mturk_file_8.csv               0                 25               0   6685   
mturk_file_9.csv               0                318               0      0   
word_list_10_long.csv          0                  0             